# `qry` — Reference

`qry` filters a DataFrame using a dict of conditions. All condition types supported:

| Syntax | Meaning |
|---|---|
| `{'col': value}` | equality |
| `{'col': [v1, v2]}` | in list |
| `{'col': ('in', [...])}` | in list (explicit) |
| `{'col': ('not in', [...])}` | exclusion |
| `{'col': ('>', n)}` | comparison operator |
| `{'col': '(a,b)'}` | open interval `a < x < b` |
| `{'col': '[a,b)'}` | half-open interval `a <= x < b` |
| `{'col': '[a,b]'}` | closed interval `a <= x <= b` |

Chain multiple `.qry()` calls to apply conditions on the same column twice.

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

## Equality and multi-value filters

In [2]:
# Single equality — keep only Dream island
# can use pandas .query but qry is more flexible for majority of use cases
(penguins
 .qry({'island': 'Dream'})
.sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
161,Chinstrap,Dream,51.3,19.9,198.0,3700.0,Male
216,Chinstrap,Dream,43.5,18.1,202.0,3400.0,Female
46,Adelie,Dream,41.1,19.0,182.0,3425.0,Male
149,Adelie,Dream,37.8,18.1,193.0,3750.0,Male
48,Adelie,Dream,36.0,17.9,190.0,3450.0,Female
208,Chinstrap,Dream,45.2,16.6,191.0,3250.0,Female
136,Adelie,Dream,35.6,17.5,191.0,3175.0,Female
45,Adelie,Dream,39.6,18.8,190.0,4600.0,Male
185,Chinstrap,Dream,51.0,18.8,203.0,4100.0,Male
167,Chinstrap,Dream,50.5,19.6,201.0,4050.0,Male


In [3]:
# List — keep Adelie and Chinstrap, aggregate per species + island
(penguins
 .qry({'species': ['Adelie', 'Chinstrap']})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
71,Adelie,Torgersen,39.7,18.4,190.0,3900.0,Male
41,Adelie,Dream,40.8,18.4,195.0,3900.0,Male
69,Adelie,Torgersen,41.8,19.4,198.0,4450.0,Male
117,Adelie,Torgersen,37.3,20.5,199.0,3775.0,Male
32,Adelie,Dream,39.5,17.8,188.0,3300.0,Female
162,Chinstrap,Dream,46.6,17.8,193.0,3800.0,Female
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,None
93,Adelie,Dream,39.6,18.1,186.0,4450.0,Male
192,Chinstrap,Dream,49.0,19.5,210.0,3950.0,Male
184,Chinstrap,Dream,42.5,16.7,187.0,3350.0,Female


In [4]:
# 'not in' — exclude Male penguins
(penguins
 .qry({'sex': ('not in', ['Male'])})
.sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
174,Chinstrap,Dream,43.2,16.6,187.0,2900.0,Female
278,Gentoo,Biscoe,43.2,14.5,208.0,4450.0,Female
274,Gentoo,Biscoe,46.5,14.4,217.0,4900.0,Female
280,Gentoo,Biscoe,45.3,13.8,208.0,4200.0,Female
22,Adelie,Biscoe,35.9,19.2,189.0,3800.0,Female
18,Adelie,Torgersen,34.4,18.4,184.0,3325.0,Female
334,Gentoo,Biscoe,46.2,14.1,217.0,4375.0,Female
130,Adelie,Torgersen,38.5,17.9,190.0,3325.0,Female
128,Adelie,Torgersen,39.0,17.1,191.0,3050.0,Female
164,Chinstrap,Dream,47.0,17.3,185.0,3700.0,Female


## Comparison operators

In [5]:
# Supported operators: >, >=, <, <=, ==, !=
# == is the default — {'col': value} and {'col': ('==', value)} are identical
# Heavy penguins (body_mass_g > 5000) across species
(penguins
 .qry({'body_mass_g': ('>', 5000)})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
305,Gentoo,Biscoe,50.8,17.3,228.0,5600.0,Male
289,Gentoo,Biscoe,50.7,15.0,223.0,5550.0,Male
297,Gentoo,Biscoe,51.1,16.3,220.0,6000.0,Male
254,Gentoo,Biscoe,49.1,14.8,220.0,5150.0,Female
331,Gentoo,Biscoe,49.8,15.9,229.0,5950.0,Male
231,Gentoo,Biscoe,49.0,16.1,216.0,5550.0,Male
269,Gentoo,Biscoe,45.2,15.8,215.0,5300.0,Male
250,Gentoo,Biscoe,47.3,15.3,222.0,5250.0,Male
237,Gentoo,Biscoe,49.2,15.2,221.0,6300.0,Male
299,Gentoo,Biscoe,45.2,16.4,223.0,5950.0,Male


In [6]:
# Combine multiple column conditions in one qry call
# Gentoo females with large bills — multi-condition dict
(penguins
 .qry({'species': 'Gentoo', 'sex': 'Female', 'bill_length_mm': ('>=', 45)})
 .select(['species', 'island', 'sex', 'bill_length_mm', 'body_mass_g'])
 .sample(10)
)

,species,island,sex,bill_length_mm,body_mass_g
296,Gentoo,Biscoe,Female,47.5,4600.0
302,Gentoo,Biscoe,Female,47.4,4725.0
298,Gentoo,Biscoe,Female,45.2,4750.0
293,Gentoo,Biscoe,Female,46.5,5200.0
342,Gentoo,Biscoe,Female,45.2,5200.0
282,Gentoo,Biscoe,Female,45.7,4400.0
317,Gentoo,Biscoe,Female,46.9,4875.0
222,Gentoo,Biscoe,Female,48.7,4450.0
226,Gentoo,Biscoe,Female,45.4,4800.0
340,Gentoo,Biscoe,Female,46.8,4850.0


## Interval notation
String intervals let you filter a range on the same column without chaining two `.qry()` calls.
`(` / `)` = exclusive bound, `[` / `]` = inclusive bound.

In [7]:
# Open interval — strictly between 3500 and 4500 g
(penguins
 .qry({'body_mass_g': '(3500,4500)'})
.sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
15,Adelie,Torgersen,36.6,17.8,185.0,3700.0,Female
146,Adelie,Dream,39.2,18.6,190.0,4250.0,Male
23,Adelie,Biscoe,38.2,18.1,185.0,3950.0,Male
85,Adelie,Dream,41.3,20.3,194.0,3550.0,Male
188,Chinstrap,Dream,47.6,18.3,195.0,3850.0,Female
276,Gentoo,Biscoe,43.8,13.9,208.0,4300.0,Female
79,Adelie,Torgersen,42.1,19.1,195.0,4000.0,Male
89,Adelie,Dream,38.9,18.8,190.0,3600.0,Female
149,Adelie,Dream,37.8,18.1,193.0,3750.0,Male
159,Chinstrap,Dream,51.3,18.2,197.0,3750.0,Male


In [8]:
# Half-open interval — bill_length_mm >= 40 and < 50
(penguins
 .qry({'bill_length_mm': '[40,50)'})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
43,Adelie,Dream,44.1,19.7,196.0,4400.0,Male
248,Gentoo,Biscoe,48.2,14.3,210.0,4600.0,Female
326,Gentoo,Biscoe,41.7,14.7,210.0,4700.0,Female
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female
37,Adelie,Dream,42.2,18.5,180.0,3550.0,Female
214,Chinstrap,Dream,45.7,17.0,195.0,3650.0,Female
281,Gentoo,Biscoe,46.2,14.9,221.0,5300.0,Male
67,Adelie,Biscoe,41.1,19.1,188.0,4100.0,Male
123,Adelie,Torgersen,41.4,18.5,202.0,3875.0,Male
317,Gentoo,Biscoe,46.9,14.6,222.0,4875.0,Female


In [9]:
# Closed interval — body_mass_g between 3500 and 4000 inclusive
(penguins
 .qry({'body_mass_g': '[3500,4000]'})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
21,Adelie,Biscoe,37.7,18.7,180.0,3600.0,Male
135,Adelie,Dream,41.1,17.5,190.0,3900.0,Male
55,Adelie,Biscoe,41.4,18.6,191.0,3700.0,Male
23,Adelie,Biscoe,38.2,18.1,185.0,3950.0,Male
6,Adelie,Torgersen,38.9,17.8,181.0,3625.0,Female
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male
76,Adelie,Torgersen,40.9,16.8,191.0,3700.0,Female
149,Adelie,Dream,37.8,18.1,193.0,3750.0,Male
195,Chinstrap,Dream,45.5,17.0,196.0,3500.0,Female
36,Adelie,Dream,38.8,20.0,190.0,3950.0,Male


In [10]:
# Interval notation also works on STRING columns — alphabetical comparison
# 'island' values: Biscoe, Dream, Torgersen — (Biscoe, Dream] means Dream only
(penguins
 .qry({'island': '(Biscoe,Dream]'})  # all islands greater than Biscoe and up to Dream
 .sample()
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
45,Adelie,Dream,39.6,18.8,190.0,4600.0,Male


## Same-column double filter: chain two `.qry()` calls
`dict` keys are unique so you can't write `{'col': ('>',a), 'col': ('<',b)}`.
The solution is to chain two `.qry()` calls, or use interval notation (shown above).

In [11]:
# Equivalent to: bill_length_mm > 40 AND bill_length_mm < 50
# (same result as '[40,50)' interval but with chained qry)
(penguins
 .qry({'bill_length_mm': ('>', 40)})
 .qry({'bill_length_mm': ('<', 50)})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
131,Adelie,Torgersen,43.1,19.2,197.0,3500.0,Male
254,Gentoo,Biscoe,49.1,14.8,220.0,5150.0,Female
53,Adelie,Biscoe,42.0,19.5,200.0,4050.0,Male
244,Gentoo,Biscoe,42.9,13.1,215.0,5000.0,Female
113,Adelie,Biscoe,42.2,19.5,197.0,4275.0,Male
322,Gentoo,Biscoe,47.2,15.5,215.0,4975.0,Female
199,Chinstrap,Dream,49.0,19.6,212.0,4300.0,Male
275,Gentoo,Biscoe,45.0,15.4,220.0,5050.0,Male
140,Adelie,Dream,40.2,17.1,193.0,3400.0,Female
257,Gentoo,Biscoe,44.4,17.3,219.0,5250.0,Male


## Real-world pipeline: filter → select → aggregate

In [12]:
# Tips: weekend dinner table of 2+, spending > $10 — average tip by smoker status
(tips
 .qry({'day': ['Sat', 'Sun'], 'time': 'Dinner', 'size': ('>=', 2), 'total_bill': ('>', 10)})
 .select(['smoker', 'tip', 'total_bill'])
 .sample(10)
)

,smoker,tip,total_bill
62,Yes,1.98,11.02
15,No,3.92,21.58
231,Yes,3.00,15.69
213,Yes,2.50,13.27
33,No,2.45,20.69
0,No,1.01,16.99
42,No,3.06,13.94
45,No,3.00,18.29
163,No,2.00,13.81
52,No,5.20,34.81
